In [1]:
from flowrec.data.vk_lsvd import VKLSVDDataConfig, VKLSVDDataModule



In [2]:
cfg = VKLSVDDataConfig(
    use_hf=True,
    split_strategy="leave_one_out",
    min_user_interactions=5,
    min_item_interactions=5,
    max_history_len=128,
    batch_size=256,
    train_all_prefixes=False,
    subsample_name='ur0.01_ir0.01',
    global_eval_target_mode='last_per_user',
    rating_mode='timespent',
    min_timespent=1.0
)

dm = VKLSVDDataModule(cfg)
dm.setup()

print(dm.summary())
batch = next(iter(dm.train_dataloader()))

/opt/anaconda3/envs/synth/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'num_users': 69268, 'num_items': 51886, 'num_train_examples': 69268, 'num_val_examples': 69262, 'num_test_examples': 69242, 'num_interactions': 3862326, 'mean_history_len': 45.12940145929191, 'mean_train_history_len': 44.242261939134956, 'mean_val_history_len': 45.13105310271144, 'mean_test_history_len': 46.01522197510182, 'post_filter_cold_items': True, 'cold_item_reference': 'observed', 'num_items_before_cold_filter': 51978, 'num_cold_items_removed': 92, 'num_train_examples_dropped_cold': 0, 'num_val_examples_dropped_cold': 6, 'num_test_examples_dropped_cold': 26, 'num_train_history_items_removed_cold': 0, 'num_val_history_items_removed_cold': 0, 'num_test_history_items_removed_cold': 6, 'dataset': 'vk_lsvd', 'split_strategy': 'leave_one_out', 'subsample_name': 'ur0.01_ir0.01', 'global_val_cutoff': None, 'global_test_cutoff': None, 'sid_len': None, 'item_vocab_size': 51887, 'post_split_filter': {'enabled': True, 'train_item_source': 'observed', 'min_history_len': 1, 'num_items_befor

In [3]:
from flowrec.item_representation.vk_lsvd import (
    VKLSVDItemEmbeddingConfig,
    VKLSVDItemEmbeddingBuilder,
)

store = VKLSVDItemEmbeddingBuilder(
    VKLSVDItemEmbeddingConfig(
        embeddings_path="../data/raw/VK-LSVD/item_embeddings.npz",
        embedding_dim=64,
    )
).build()

store.save("../artifacts/item_embeddings/vk_lsvd_embeddings.pt")

In [4]:
from flowrec.item_representation.semantic_id import (
    RQVAEConfig,
    RQVAETrainConfig,
    RQVAEItemTokenizerConfig,
    RQVAEItemTokenizer,
)


# Keep only the items that remain in the final dataset after preprocessing, then
# train the RQ-VAE tokenizer on exactly those content embeddings.
store_in_data = store.subset(dm.item_ids_in_data())
print(f"{store_in_data.num_items} / {store.num_items} items remain in the final dataset")

tokenizer = RQVAEItemTokenizer(
    RQVAEItemTokenizerConfig(
        rqvae=RQVAEConfig(
            input_dim=store_in_data.embeddings.shape[1],
            latent_dim=32,
            num_codebooks=3,
            codebook_size=256,
        ),
        train=RQVAETrainConfig(epochs=10, device="cpu"),
        normalize="l2",
        collision_policy="append",
    )
)

tokenizer.fit(store_in_data.embeddings, store_in_data.item_ids)
tokenizer.save("../artifacts/tokenizers/vk_lsvd_rqvae")

semantic_id_mapping = tokenizer.semantic_id_store.to_plain_dict()

51886 / 19627601 items remain in the final dataset
Initializing RQ-VAE codebooks from data...


rqvae epoch 1/10: 100%|██████████| 49/49 [00:05<00:00,  8.33it/s, loss=0.0113, recon_loss=0.0112, vq_loss=0.000195, dead_codes_l0=3, dead_codes_l1=0, dead_codes_l2=1] 


{"epoch": 1, "step": 49, "train_dead_codes_l0": 61.957072, "train_dead_codes_l1": 2.742189, "train_dead_codes_l2": 2.786578, "train_loss": 0.012571, "train_recon_loss": 0.012429, "train_vq_loss": 0.000142, "val_loss": 0.011363, "val_recon_loss": 0.011173, "val_vq_loss": 0.00019}


rqvae epoch 2/10: 100%|██████████| 49/49 [00:05<00:00,  8.62it/s, loss=0.0105, recon_loss=0.0103, vq_loss=0.000172, dead_codes_l0=0, dead_codes_l1=3, dead_codes_l2=1]


{"epoch": 2, "step": 98, "train_dead_codes_l0": 0.540128, "train_dead_codes_l1": 1.795099, "train_dead_codes_l2": 2.973545, "train_loss": 0.010954, "train_recon_loss": 0.010789, "train_vq_loss": 0.000164, "val_loss": 0.010483, "val_recon_loss": 0.010307, "val_vq_loss": 0.000175}


rqvae epoch 3/10: 100%|██████████| 49/49 [00:05<00:00,  9.52it/s, loss=0.00992, recon_loss=0.00946, vq_loss=0.00045, dead_codes_l0=0, dead_codes_l1=0, dead_codes_l2=0] 


... (11 outputs left out)

rqvae epoch 9/10: 100%|██████████| 49/49 [00:04<00:00, 10.20it/s, loss=0.00757, recon_loss=0.00693, vq_loss=0.000645, dead_codes_l0=0, dead_codes_l1=0, dead_codes_l2=0]


{"epoch": 9, "step": 441, "train_dead_codes_l0": 0.0, "train_dead_codes_l1": 0.0, "train_dead_codes_l2": 0.0, "train_loss": 0.007959, "train_recon_loss": 0.007352, "train_vq_loss": 0.000607, "val_loss": 0.00791, "val_recon_loss": 0.007228, "val_vq_loss": 0.000682}


rqvae epoch 10/10: 100%|██████████| 49/49 [00:04<00:00,  9.86it/s, loss=0.0078, recon_loss=0.007, vq_loss=0.000798, dead_codes_l0=0, dead_codes_l1=0, dead_codes_l2=0]   


{"epoch": 10, "step": 490, "train_dead_codes_l0": 0.0, "train_dead_codes_l1": 0.0, "train_dead_codes_l2": 0.0, "train_loss": 0.007824, "train_recon_loss": 0.007104, "train_vq_loss": 0.000721, "val_loss": 0.007839, "val_recon_loss": 0.007058, "val_vq_loss": 0.000781}


In [5]:
# Attach the trained semantic IDs to the datamodule that was already built and
# preprocessed above, instead of constructing a second instance from a saved
# semantic_ids.json path.
dm.add_semantic_ids(semantic_ids=semantic_id_mapping)

batch = next(iter(dm.train_dataloader()))
print(batch["history_token_ids"].shape)
print(batch["target_token_ids"].shape)

torch.Size([256, 128, 4])
torch.Size([256, 4])
